In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/marthadimgba/spy-agency-dataset-2024/securityclearance.csv
/kaggle/input/datasets/marthadimgba/spy-agency-dataset-2024/affiliationrel.csv
/kaggle/input/datasets/marthadimgba/spy-agency-dataset-2024/languagerel.csv
/kaggle/input/datasets/marthadimgba/spy-agency-dataset-2024/mission.csv
/kaggle/input/datasets/marthadimgba/spy-agency-dataset-2024/agent.csv
/kaggle/input/datasets/marthadimgba/spy-agency-dataset-2024/team.csv
/kaggle/input/datasets/marthadimgba/spy-agency-dataset-2024/skill.csv
/kaggle/input/datasets/marthadimgba/spy-agency-dataset-2024/affiliation.csv
/kaggle/input/datasets/marthadimgba/spy-agency-dataset-2024/language.csv
/kaggle/input/datasets/marthadimgba/spy-agency-dataset-2024/teamrel.csv
/kaggle/input/datasets/marthadimgba/spy-agency-dataset-2024/skillrel.csv
/kaggle/input/datasets/smalleyes0m364/netsec-and-radio/netsec_sft_example.jsonl
/kaggle/input/datasets/smalleyes0m364/netsec-and-radio/radio_signals_sft_train.jsonl
/kaggle/input/datasets/

In [2]:
# 1) Start runtime sanity checks + path discovery
!nvidia-smi
!find /kaggle/input -maxdepth 7 -type f | grep -Ei "config.json|netsec_sft_train.jsonl|netsec_sft_val.jsonl" | head -200


Tue Mar 10 01:43:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 2) Install stable deps (no trl)
!pip -q install -U \
  "transformers==4.46.3" \
  "peft==0.12.0" \
  "accelerate==0.34.2" \
  "datasets==2.21.0" \
  "bitsandbytes>=0.46.1" \
  "safetensors==0.4.5" \
  "sentencepiece==0.2.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.8/434.8 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 54.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [4]:
# 3) Training (auto-finds model + dataset files)
import os, json, glob, torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

# auto-find
train_candidates = glob.glob("/kaggle/input/**/netsec_sft_train.jsonl", recursive=True)
val_candidates = glob.glob("/kaggle/input/**/netsec_sft_val.jsonl", recursive=True)
model_cfg_candidates = [p for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "qwen" in p.lower() and "coder" in p.lower()]

assert train_candidates, "No netsec_sft_train.jsonl found"
assert val_candidates, "No netsec_sft_val.jsonl found"
assert model_cfg_candidates, "No Qwen Coder config.json found"

TRAIN_JSONL = train_candidates[0]
VAL_JSONL = val_candidates[0]
BASE_MODEL = os.path.dirname(model_cfg_candidates[0])
OUT_DIR = "/kaggle/working/qwen-coder-netsec-lora"

print("BASE_MODEL:", BASE_MODEL)
print("TRAIN:", TRAIN_JSONL)
print("VAL:", VAL_JSONL)

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

def to_text(row):
    instr = str(row.get("instruction","")).strip()
    inp = str(row.get("input","")).strip()
    ctx = str(row.get("context","")).strip()
    out = str(row.get("output","")).strip()
    user = "\n\n".join([x for x in [
        f"Instruction:\n{instr}" if instr else "",
        f"Input:\n{inp}" if inp else "",
        f"Context:\n{ctx}" if ctx else "",
    ] if x])
    msgs = [
        {"role":"system","content":"You are a defensive network security assistant. Be precise, concise, and safe."},
        {"role":"user","content":user},
        {"role":"assistant","content":out},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

train_rows = load_jsonl(TRAIN_JSONL)
val_rows = load_jsonl(VAL_JSONL)

train_ds = Dataset.from_dict({"text":[to_text(r) for r in train_rows]})
val_ds = Dataset.from_dict({"text":[to_text(r) for r in val_rows]})

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=768)

train_ds = train_ds.map(tok, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tok, batched=True, remove_columns=["text"])

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=1,                  # start small
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    logging_steps=10,
    save_steps=200,
    eval_steps=200,
    save_total_limit=2,
    fp16=True,
    evaluation_strategy="steps",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Saved:", OUT_DIR)


2026-03-10 01:43:59.724228: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773107039.880536      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773107039.925736      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773107040.309943      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773107040.309980      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773107040.309984      24 computation_placer.cc:177] computation placer alr

BASE_MODEL: /kaggle/input/models/qwen-lm/qwen2.5-coder/transformers/3b-instruct/1
TRAIN: /kaggle/input/datasets/smalleyes0m364/netsec-and-radio/netsec_sft_train.jsonl
VAL: /kaggle/input/datasets/smalleyes0m364/netsec-and-radio/netsec_sft_val.jsonl


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Map:   0%|          | 0/2999 [00:00<?, ? examples/s]

Map:   0%|          | 0/333 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss,Validation Loss


Saved: /kaggle/working/qwen-coder-netsec-lora


In [5]:
# 4) Verify adapter
!ls -la /kaggle/working/qwen-coder-netsec-lora


total 132540
drwxr-xr-x 3 root root      4096 Mar 10 02:09 .
drwxr-xr-x 3 root root      4096 Mar 10 01:44 ..
-rw-r--r-- 1 root root       772 Mar 10 02:09 adapter_config.json
-rw-r--r-- 1 root root 119801528 Mar 10 02:09 adapter_model.safetensors
-rw-r--r-- 1 root root       605 Mar 10 02:09 added_tokens.json
drwxr-xr-x 2 root root      4096 Mar 10 02:09 checkpoint-187
-rw-r--r-- 1 root root   1671853 Mar 10 02:09 merges.txt
-rw-r--r-- 1 root root      5143 Mar 10 02:09 README.md
-rw-r--r-- 1 root root       613 Mar 10 02:09 special_tokens_map.json
-rw-r--r-- 1 root root      7306 Mar 10 02:09 tokenizer_config.json
-rw-r--r-- 1 root root  11421994 Mar 10 02:09 tokenizer.json
-rw-r--r-- 1 root root   2776833 Mar 10 02:09 vocab.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
